# Account-Level Loss Compression — lite

Compresses an account-level loss table, reconstructs the per-account metrics, and reports marginal
impact at a return period.

**Two possible inputs.** Set `INPUT_MODE` in the config cell:

| mode | you provide | §0 does |
|---|---|---|
| `"PLT"` | `account_plt`, `portfolio_plt` | nothing — reads them straight |
| `"ELT"` | `elt` (account-level) | simulates both PLTs from it |

Everything from §1 onwards is identical either way.

| table | columns |
|---|---|
| `elt` | `event_id, accnt_no, rate, mean_loss, sd_indep, sd_corr, exposure` |
| `account_plt` | `accnt_no, event_id, year_id, loss_date, loss` |
| `portfolio_plt` | `event_id, year_id, loss_date, loss` |

Both bases are handled: **AEP** (annual sum) and **OEP** (largest single occurrence). They need
different reconstruction paths, which is most of what §4 is about.

In [ ]:
import numpy as np, pandas as pd
from scipy import stats, sparse

Q_RETAIN  = 0.98      # store the worst 2% of years exactly
ALPHA     = 0.99      # metrics reported at this level
BLOCK     = 50_000    # occurrences per block in the OEP scan (sets peak memory)
OCC_KEY   = ["year_id", "event_id", "loss_date"]

INPUT_MODE = "ELT"     # "ELT" -> simulate the PLTs from an account ELT
                       # "PLT" -> read two PLTs directly, skip section 0
T_YEARS    = 20_000    # trial years (ELT mode; in PLT mode it is inferred)
SEED       = 42        # engine seed; the same seed reproduces the tables exactly

assert ALPHA >= Q_RETAIN, "a metric level outside the retained region is not covered"
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

## 0 · ELT input — simulate the PLTs

Skip this section entirely if you already have the two PLTs (`INPUT_MODE = "PLT"`).

### Why the portfolio ELT→YLT algorithm cannot be reused

| | portfolio grain | account grain |
|---|---|---|
| one ELT row is | one event | one `(event, account)` pair |
| frequency | Poisson superposition over rows, then sample which row | Poisson per **event**, then loop accounts within |
| severity | `rng.beta(a, b)` per row, independent | `beta.ppf(norm.cdf(shared + private))` |
| `sd_corr` | folded into `sd_tot`, treated as independent | drives a shock **shared** across the footprint |

Feeding an account ELT to the portfolio algorithm fails *silently*: pooling `rate` across
`(event, account)` rows inflates frequency by roughly the accounts-per-event count, and sampling one
row per occurrence makes accounts independent — one hurricane hitting eight accounts becomes eight
separate occurrences hitting one account each. The portfolio tail collapses and every diversification
and marginal-impact number below is wrong, with no error raised.

### Severity convention

    sd_tot = sd_indep + sd_corr                       (linear, RMS-style)
    damage ratio ~ Beta(mean = mean_loss/exposure, sd = sd_tot/exposure)
    loss = damage ratio x exposure

Correlation comes from a Gaussian copula:

    u    = Phi( w_corr * z_shared + w_indep * z_private )
    loss = BetaPPF(u; a, b) * exposure

with `w_corr, w_indep` proportional to `sd_corr, sd_indep` and normalised so
`w_corr^2 + w_indep^2 = 1`. The Beta **marginal** is identical to `rng.beta(a, b)` — a quantile
transform of a uniform *is* a Beta. Only the dependence changes.

In [4]:
REQUIRED = {"event_id", "accnt_no", "rate", "mean_loss",
            "sd_indep", "sd_corr", "exposure"}

# --- the engine (only used when INPUT_MODE == 'ELT') ---

def beta_params_from_moments(m, s):
    """Method-of-moments Beta from mean ratio `m` and sd ratio `s`, both in (0, 1).
    Returns (a, b, capped)."""
    m = np.asarray(m, dtype=float)
    s = np.asarray(s, dtype=float)
    s_max  = np.sqrt(m * (1.0 - m))
    capped = s >= s_max
    s  = np.where(capped, np.nextafter(s_max, 0.0) * (1 - 1e-9), s)
    nu = m * (1.0 - m) / s ** 2 - 1.0
    return m * nu, (1.0 - m) * nu, capped

def elt_to_account_plt(elt, n_years, seed=42, secondary=True, days_in_year=365,
                       drop_zero=True, zero_tol=0.0, verbose=True):
    """Simulate an account-level PLT from an account-level ELT.

    Parameters
    ----------
    elt          : DataFrame with the REQUIRED columns, one row per (event_id, accnt_no)
    n_years      : number of trial years
    seed         : root seed; the same seed reproduces the table bit for bit
    secondary    : if False, use the deterministic mean_loss (accounts still co-occur)
    days_in_year : calendar length used to place occurrences on dates
    drop_zero    : drop rows with loss <= zero_tol, mirroring a vendor extract that does
                   not write sub-threshold losses; set zero_tol to your real threshold
    verbose      : print the Beta-capping warning if any occurrence hits the bound

    Returns
    -------
    DataFrame: accnt_no, event_id, year_id (1-based), loss_date, loss
    """
    missing = REQUIRED - set(elt.columns)
    if missing:
        raise ValueError(f"account ELT missing columns: {sorted(missing)}")
    if n_years < 1:
        raise ValueError("n_years must be >= 1")

    # --- rate is a property of the EVENT, not of the (event, account) pair ---
    rbe = elt.groupby("event_id")["rate"].agg(["min", "max", "first"])
    if not np.allclose(rbe["min"], rbe["max"]):
        bad = rbe.index[~np.isclose(rbe["min"], rbe["max"])].tolist()
        raise ValueError(f"rate varies within event_id for events {bad[:5]}"
                         f"{' ...' if len(bad) > 5 else ''}; it must be constant across accounts")

    rng = np.random.default_rng(seed)

    # stable order == reproducible RNG consumption
    elt    = elt.sort_values(["event_id", "accnt_no"], kind="stable")
    events = rbe.index.to_numpy()
    rates  = rbe["first"].to_numpy(dtype=float)

    frames, n_capped = [], 0

    for event_id, rate in zip(events, rates):
        legs = elt[elt["event_id"] == event_id]

        # --- 1. frequency: Poisson per EVENT, not per row -------------------
        counts = rng.poisson(rate, size=n_years)
        n_occ  = int(counts.sum())
        if n_occ == 0:
            continue
        year = np.repeat(np.arange(1, n_years + 1), counts)

        # --- 2. dates distinct within (year, event) -------------------------
        start  = np.concatenate([[0], np.cumsum(counts)])
        within = np.arange(n_occ) - np.repeat(start[:-1], counts)
        base   = rng.integers(1, days_in_year + 1, size=n_years)
        day    = 1 + (base[year - 1] - 1 + within) % days_in_year

        # --- 3. THE COMMON SHOCK: one draw per occurrence, shared by every --
        #        account exposed to it. This is the whole mechanism.
        z_shared = rng.standard_normal(n_occ) if secondary else None

        # --- 4. per account in the footprint --------------------------------
        for leg in legs.itertuples(index=False):
            mu, E = float(leg.mean_loss), float(leg.exposure)

            if not secondary or E <= 0:
                loss = np.full(n_occ, mu, dtype=float)
            else:
                m = mu / E
                s = (float(leg.sd_indep) + float(leg.sd_corr)) / E
                if s <= 0 or m <= 0 or m >= 1:
                    loss = np.full(n_occ, mu, dtype=float)          # degenerate -> mean
                else:
                    a, b, capped = beta_params_from_moments(m, s)
                    n_capped += int(np.atleast_1d(capped).sum())

                    w = np.hypot(float(leg.sd_corr), float(leg.sd_indep))
                    if w <= 0:
                        w_corr, w_indep = 0.0, 1.0
                    else:
                        w_corr, w_indep = float(leg.sd_corr) / w, float(leg.sd_indep) / w

                    z    = w_corr * z_shared + w_indep * rng.standard_normal(n_occ)
                    loss = stats.beta.ppf(stats.norm.cdf(z), a, b) * E

            frames.append(pd.DataFrame({"accnt_no": leg.accnt_no, "event_id": event_id,
                                        "year_id": year, "loss_date": day, "loss": loss}))

    if not frames:
        return pd.DataFrame(columns=["accnt_no", "event_id", "year_id", "loss_date", "loss"])

    out = pd.concat(frames, ignore_index=True)
    if drop_zero:
        out = out[out["loss"] > zero_tol].reset_index(drop=True)
    if verbose and n_capped:
        print(f"[warn] sd capped at the Beta bound for {n_capped} account-events")

    return (out.sort_values(["year_id", "event_id", "loss_date", "accnt_no"])
               .reset_index(drop=True))

def aggregate_to_portfolio_plt(account_plt):
    """Sum away the account dimension: one row per occurrence."""
    return (account_plt.groupby(["year_id", "event_id", "loss_date"], as_index=False)["loss"]
            .sum().sort_values(["year_id", "event_id", "loss_date"]).reset_index(drop=True))


def validate_account_plt(account_plt, elt, n_years, z_tol=4.0):
    """Per-account AAL against the ELT analytic value, in units of Monte Carlo error."""
    analytic = elt.assign(_a=elt["rate"] * elt["mean_loss"]).groupby("accnt_no")["_a"].sum()
    var = (elt.assign(_v=elt["rate"] * (elt["mean_loss"] ** 2
                                        + (elt["sd_indep"] + elt["sd_corr"]) ** 2))
              .groupby("accnt_no")["_v"].sum())
    sim = (account_plt.groupby("accnt_no")["loss"].sum()
           .reindex(analytic.index).fillna(0.0) / n_years)
    se  = np.sqrt(var / n_years)
    res = pd.DataFrame({"aal_analytic": analytic, "aal_simulated": sim,
                        "se": se, "z": (sim - analytic) / se})
    res["ok"] = res["z"].abs() < z_tol
    return res

### The ELT

Replace this cell with your own load. If your columns are named differently (`EventId`,
`AccountId`, `MeanLoss`, …) rename them first — `itertuples` accesses by attribute, so
`leg.mean_loss` would otherwise need to become `leg.MeanLoss`.

In [5]:
if INPUT_MODE == "ELT":
    _r0      = np.random.default_rng(7)
    _n_ev    = 30
    accounts = [f"ACC-{i:02d}" for i in range(1, 9)]
    _expo    = dict(zip(accounts, [2500, 1800, 1200, 900, 3000, 600, 1500, 2200]))
    _rates   = dict(zip(range(1, _n_ev + 1), 0.30 * 0.88 ** np.arange(_n_ev)))

    _recs = []
    for a in accounts:
        _hit = np.arange(1, _n_ev + 1)[_r0.random(_n_ev) < 0.45]        # this account's footprint
        for e, m in zip(_hit, np.geomspace(0.002, _r0.uniform(0.05, 0.30), len(_hit))):
            ml = m * _expo[a]
            _recs.append({"event_id": e, "accnt_no": a, "rate": _rates[e], "mean_loss": ml,
                          "sd_indep": 0.40 * ml, "sd_corr": 0.30 * ml, "exposure": float(_expo[a])})
    elt = pd.DataFrame(_recs).sort_values(["event_id", "accnt_no"]).reset_index(drop=True)

    print(f"account ELT: {len(elt)} rows | {elt.event_id.nunique()} events | "
          f"{elt.accnt_no.nunique()} accounts | "
          f"{elt.groupby('event_id').size().mean():.1f} accounts per event")
    print(f"portfolio AAL implied by the ELT = {(elt.rate * elt.mean_loss).sum():,.2f}")
    display(elt.head())

account ELT: 106 rows | 28 events | 8 accounts | 3.8 accounts per event
portfolio AAL implied by the ELT = 215.25


,event_id,accnt_no,rate,mean_loss,sd_indep,sd_corr,exposure
0,1,ACC-02,0.300,3.600,1.440,1.080,"1,800.000"
1,1,ACC-07,0.300,3.000,1.200,0.900,"1,500.000"
2,2,ACC-02,0.264,5.019,2.008,1.506,"1,800.000"
3,2,ACC-03,0.264,2.400,0.960,0.720,"1,200.000"
4,2,ACC-04,0.264,1.800,0.720,0.540,900.000


In [ ]:
# importing ELT from a database instead of simulating it
'''if INPUT_MODE == "ELT":
    import sqlalchemy as sa
    eng = sa.create_engine("mssql+pyodbc://SERVER/DB?driver=ODBC+Driver+17+for+SQL+Server")

    elt = pd.read_sql("""
        SELECT EventId    AS event_id,
               AccountId  AS accnt_no,
               Rate       AS rate,
               MeanLoss   AS mean_loss,
               StdDevI    AS sd_indep,
               StdDevC    AS sd_corr,
               Exposure   AS exposure
        FROM   dbo.AccountELT
        WHERE  AnalysisId = ?
    """, eng, params=(analysis_id,))

    accounts = sorted(elt.accnt_no.unique())
    display(elt.head())'''

### Simulate

`year_id` is shifted to 0-based because everything downstream indexes arrays of length `T_YEARS`
directly.

In [6]:
if INPUT_MODE == "ELT":
    account_plt = elt_to_account_plt(elt, n_years=T_YEARS, seed=SEED)
    account_plt["year_id"] -= 1                       # engine is 1-based; arrays are 0-based
    portfolio_plt = (account_plt.groupby(OCC_KEY, as_index=False)["loss"].sum())
    print(f"simulated {len(account_plt):,} account rows -> {len(portfolio_plt):,} occurrences")

simulated 206,737 account rows -> 47,027 occurrences


### Engine checks

Run these once on a new ELT. The last one is what matters — the others can all pass while the engine
is quietly producing independent accounts.

In [7]:
if INPUT_MODE == "ELT":
    # 1. AAL closes against the ELT (z-scores, not flat percentages)
    display(validate_account_plt(account_plt, elt, T_YEARS).round(3))

    # 2. occurrence key unique -- OEP needs this
    assert not portfolio_plt.duplicated(OCC_KEY).any()
    print("occurrence key unique: True")

    # 3. same seed -> identical table
    _again = elt_to_account_plt(elt, n_years=T_YEARS, seed=SEED, verbose=False)
    assert np.array_equal(np.sort(account_plt.loss.to_numpy()), np.sort(_again.loss.to_numpy()))
    print("reproducible under the same seed: True"); del _again

    # 4. accounts MUST co-move within an event -- this is what sd_corr buys
    _ev  = portfolio_plt.event_id.value_counts().index[0]
    _piv = (account_plt[account_plt.event_id == _ev]
            .pivot_table(index=["year_id", "loss_date"], columns="accnt_no", values="loss"))
    print(f"\nwithin-event correlation of account losses, event {_ev}:")
    display(_piv.corr().round(3))
    print("If these are ~0 the shared shock is not wired through: the portfolio tail would be far")
    print("too thin and every number below would be wrong while looking reasonable.")

,aal_analytic,aal_simulated,se,z,ok
accnt_no,,,,,
ACC-01,31.893,31.316,0.635,-0.909,True
ACC-02,36.566,36.088,0.565,-0.846,True
ACC-03,17.740,17.436,0.270,-1.124,True
ACC-04,12.948,12.685,0.219,-1.201,True
ACC-05,58.210,58.011,0.918,-0.217,True
ACC-06,5.820,5.837,0.071,0.245,True
ACC-07,33.213,32.791,0.524,-0.806,True
ACC-08,18.863,18.732,0.502,-0.260,True


occurrence key unique: True
reproducible under the same seed: True

within-event correlation of account losses, event 1:


accnt_no,ACC-02,ACC-07
accnt_no,,
ACC-02,1.000,0.350
ACC-07,0.350,1.000


If these are ~0 the shared shock is not wired through: the portfolio tail would be far
too thin and every number below would be wrong while looking reasonable.


## 1 · The input tables

In `INPUT_MODE = "PLT"` this is where you read your two tables. In `"ELT"` mode they already exist
from §0 and this cell only derives the bookkeeping.

Casting `accnt_no` to `category` and the ids to `int32` roughly halves memory on a 10M-row table and
speeds up every groupby below.

In [ ]:
# importing from SQL database instead of simulating the PLT


In [ ]:
if INPUT_MODE == "PLT":
    # ---- replace with your own load -------------------------------------
    # account_plt   = pd.read_parquet("account_plt.parquet")
    # portfolio_plt = pd.read_parquet("portfolio_plt.parquet")
    
    '''
    account_plt = pd.read_sql("""
        SELECT AccountId AS accnt_no, EventId AS event_id,
               TrialId   AS year_id,  LossDate AS loss_date, Loss AS loss
        FROM   dbo.AccountPLT WHERE AnalysisId = ?
    """, eng, params=(analysis_id,))

    portfolio_plt = account_plt.groupby(OCC_KEY, as_index=False)["loss"].sum()'''
    raise NotImplementedError("point INPUT_MODE='PLT' at your own read_parquet calls")

accounts = sorted(account_plt["accnt_no"].unique())
A_IDX    = {a: i for i, a in enumerate(accounts)}
N_ACC    = len(accounts)
T_YEARS  = int(account_plt["year_id"].max()) + 1

assert not portfolio_plt.duplicated(OCC_KEY).any(), "occurrence key is not unique"
assert np.isclose(account_plt.loss.sum(), portfolio_plt.loss.sum()), "the two PLTs disagree"

print(f"{len(account_plt):,} account rows | {len(portfolio_plt):,} occurrences | "
      f"{N_ACC} accounts | {T_YEARS:,} years")
print(f"accounts per occurrence {len(account_plt)/len(portfolio_plt):.2f}")
display(account_plt.head())

206,737 account rows | 47,027 occurrences | 8 accounts | 20,000 years
accounts per occurrence 4.40


,accnt_no,event_id,year_id,loss_date,loss
0,ACC-02,1,0,237,2.582
1,ACC-07,1,0,237,2.415
2,ACC-02,6,0,75,13.993
3,ACC-03,6,0,75,7.483
4,ACC-04,6,0,75,5.379


## 2 · Compress

**Step 1 — the two annual metrics.** `Y[t]` = total loss in year `t` (AEP), `X[t]` = largest single
occurrence in year `t` (OEP).

**Step 2 — pick the years to keep.** The worst `(1−q)·T` by `Y`, **union** the worst `(1−q)·T` by `X`.
The union matters: a year with one enormous event ranks high on OEP but may not on AEP, and a year of
several moderate events ranks high on AEP but not OEP. Ranking is on the **year**, never on
individual occurrences — a bad year can be several moderate events, none individually large.

**Step 3 — split.** Retained years: every account row kept as-is. All other years ("the body"): rows
discarded, replaced by one share vector per event.

The share is `Σ losses to account a ÷ Σ portfolio loss` over all body occurrences of that event —
**sum first, then divide**. Averaging per-row ratios instead would bias against accounts that take a
larger slice of the larger events.

In [9]:
# --- 1. annual metrics -----------------------------------------------------
Y = np.bincount(portfolio_plt.year_id, weights=portfolio_plt.loss, minlength=T_YEARS)   # AEP
X = (portfolio_plt.groupby("year_id")["loss"].max()
     .reindex(range(T_YEARS), fill_value=0.0).to_numpy())                               # OEP

# --- 2. retained years: union of the AEP and OEP tails --------------------
k        = int(np.ceil((1 - Q_RETAIN) * T_YEARS))
rank     = lambda v, n: np.argsort(-v, kind="stable")[:n]
keep_yrs = np.union1d(rank(Y, k), rank(X, k))
is_kept  = np.zeros(T_YEARS, bool); is_kept[keep_yrs] = True

# --- 3a. retained rows, verbatim ------------------------------------------
tail_store = account_plt[is_kept[account_plt.year_id.to_numpy()]].copy()

# --- 3b. body rows -> one share vector per event --------------------------
body_acct = account_plt[~is_kept[account_plt.year_id.to_numpy()]]
body_port = portfolio_plt[~is_kept[portfolio_plt.year_id.to_numpy()]]

numer  = body_acct.groupby(["event_id", "accnt_no"])["loss"].sum()      # sum first ...
denom  = body_port.groupby("event_id")["loss"].sum()
shares = (numer / denom).unstack(fill_value=0.0).reindex(columns=accounts).fillna(0.0)  # ... then divide

assert np.allclose(shares.sum(axis=1), 1.0), "each event's shares must sum to 1"
print(f"AEP tail {k:,} years | OEP tail {k:,} | union {len(keep_yrs):,} "
      f"({len(keep_yrs)/k:.2f}x a single metric)")
print(f"stored {len(tail_store):,} rows = {100*len(tail_store)/len(account_plt):.1f}% of the "
      f"account table, plus a {shares.shape[0]} x {shares.shape[1]} share table")
display(shares.head())

AEP tail 401 years | OEP tail 401 | union 461 (1.15x a single metric)
stored 7,199 rows = 3.5% of the account table, plus a 28 x 8 share table


accnt_no,ACC-01,ACC-02,ACC-03,ACC-04,ACC-05,ACC-06,ACC-07,ACC-08
event_id,,,,,,,,
1,0.000,0.546,0.000,0.000,0.000,0.000,0.454,0.000
2,0.000,0.201,0.095,0.072,0.240,0.048,0.167,0.176
3,0.000,0.266,0.124,0.000,0.333,0.063,0.214,0.000
4,0.140,0.000,0.126,0.073,0.367,0.066,0.227,0.000
5,0.115,0.144,0.092,0.056,0.283,0.050,0.161,0.100


## 3 · What reconstruction means

For a retained year, read the stored row. For a body occurrence with event `e` and portfolio loss
`L`:

    loss(account a) = share[e, a] × L

Note this is *denser* than the truth: the share vector is non-zero for every account ever touched by
event `e`, so reconstruction assigns a loss to all of them on every occurrence, where the real table
only has rows for the accounts actually hit that time. Check the inflation on your own data before
materialising anything row-level:

In [10]:
per_occ   = len(account_plt) / len(portfolio_plt)
per_event = account_plt.groupby("event_id")["accnt_no"].nunique().mean()
print(f"accounts per occurrence {per_occ:.2f} | accounts per event {per_event:.2f} "
      f"-> row inflation {per_event/per_occ:.2f}x")
print(f"a row-level reconstruction would be ~{len(portfolio_plt)*per_event:,.0f} rows "
      f"vs {len(account_plt):,} true")
print("\nThe metrics below need per-account ANNUAL values, so the row-level table is never built.")

accounts per occurrence 4.40 | accounts per event 3.79 -> row inflation 0.86x
a row-level reconstruction would be ~178,031 rows vs 206,737 true

The metrics below need per-account ANNUAL values, so the row-level table is never built.


## 4 · Reconstruct the annual metrics

The two bases need different routes, and this is the part that matters at scale.

**AEP — annual sum.** Summing over occurrences is exactly a matrix product: group body losses by
`(year, event)`, then multiply that sparse matrix by the share table. Output is `T × n_accounts`
directly; nothing occurrence-level is ever materialised.

**OEP — annual max.** A max cannot be recovered from sums, so per-occurrence values are needed. But
they are consumed in **blocks**: build `BLOCK × n_accounts` at a time and fold it into a running max.
Peak memory is `BLOCK × n_accounts × 8` bytes no matter how many occurrences there are — 200 MB at
50,000 × 500. This is the one place blocking is genuinely required.

Retained years are added afterwards straight from the stored rows, so they stay exact.

In [11]:
# ---------- AEP: annual SUM per account -----------------------------------
AEP = np.zeros((T_YEARS, N_ACC))
by  = body_port.groupby(["year_id", "event_id"], observed=True)["loss"].sum().reset_index()
ec  = shares.index.get_indexer(by.event_id)
assert (ec >= 0).all(), "a body occurrence has an event absent from the share table"
S   = sparse.csr_matrix((by.loss.to_numpy(), (by.year_id.to_numpy(), ec)),
                        shape=(T_YEARS, len(shares)))
AEP += S @ shares.to_numpy()                                     # body, in one product
np.add.at(AEP, (tail_store.year_id.to_numpy(), tail_store.accnt_no.map(A_IDX).to_numpy()),
          tail_store.loss.to_numpy())                            # retained years, exact

# ---------- OEP: annual MAX per account -----------------------------------
OEP  = np.zeros((T_YEARS, N_ACC))
Sarr = shares.to_numpy()
ecb  = shares.index.get_indexer(body_port.event_id)
yrb, lsb = body_port.year_id.to_numpy(), body_port.loss.to_numpy()
for s in range(0, len(body_port), BLOCK):
    sl = slice(s, s + BLOCK)
    np.maximum.at(OEP, yrb[sl], Sarr[ecb[sl]] * lsb[sl][:, None])   # only BLOCK x N_ACC alive
np.maximum.at(OEP, (tail_store.year_id.to_numpy(), tail_store.accnt_no.map(A_IDX).to_numpy()),
              tail_store.loss.to_numpy())                            # retained years, exact

print(f"AEP via sparse product | OEP via {len(range(0, len(body_port), BLOCK))} blocks of "
      f"{BLOCK:,} (peak {BLOCK*N_ACC*8/1e6:.0f} MB)")

AEP via sparse product | OEP via 1 blocks of 50,000 (peak 3 MB)


In [12]:
# ---------- the truth, for comparison only --------------------------------
ay, aa, al = (account_plt.year_id.to_numpy(), account_plt.accnt_no.map(A_IDX).to_numpy(),
              account_plt.loss.to_numpy())
AEP_T = np.zeros((T_YEARS, N_ACC)); np.add.at(AEP_T, (ay, aa), al)
OEP_T = np.zeros((T_YEARS, N_ACC)); np.maximum.at(OEP_T, (ay, aa), al)
print(f"AEP totals reconcile: {np.allclose(AEP_T.sum(1), Y)} | "
      f"OEP portfolio max reconciles: {np.allclose(portfolio_plt.groupby('year_id').loss.max().reindex(range(T_YEARS), fill_value=0), X)}")

AEP totals reconcile: True | OEP portfolio max reconciles: True


## 5 · The metrics

- **AAL** — mean annual loss
- **VaR / TVaR** at 99%, standalone: on the account's *own* worst years
- **co-TVaR** — the allocation metric, read in the years where the **portfolio** is worst

co-TVaR is defined differently on the two bases, and the OEP one matters:

- *AEP basis*: the account's annual sum, averaged over the portfolio's worst `Y` years.
- *OEP basis*: the account's loss **on the single largest portfolio occurrence** of each worst `X`
  year — not the account's own annual max. Taking each account's own max would not sum back to the
  portfolio OEP TVaR, because different accounts peak on different occurrences. Reading the one
  occurrence keeps the allocation additive.

In [13]:
m         = int(round((1 - ALPHA) * T_YEARS))
aep_tail  = rank(Y, m)                          # portfolio's worst years, AEP basis
oep_tail  = rank(X, m)                          # ... OEP basis

# the single largest portfolio occurrence in each OEP-tail year
biggest = (portfolio_plt.loc[portfolio_plt.groupby("year_id")["loss"].idxmax()]
           .set_index("year_id").loc[oep_tail].reset_index())
big_key = pd.MultiIndex.from_frame(biggest[OCC_KEY])

def oep_cotvar(acct_rows):
    hit = acct_rows.set_index(OCC_KEY)
    hit = hit[hit.index.isin(big_key)]
    return hit.groupby("accnt_no")["loss"].sum().reindex(accounts).fillna(0.0).to_numpy() / m

# reconstructed: retained years are stored, so those occurrences are exact; any that fell in
# the body are rebuilt from the share table
in_store = biggest.set_index(OCC_KEY).index.isin(tail_store.set_index(OCC_KEY).index)
rec_occ  = oep_cotvar(tail_store[tail_store.set_index(OCC_KEY).index.isin(big_key)].reset_index(drop=True))
miss     = biggest[~in_store]
if len(miss):
    rec_occ = rec_occ + (shares.to_numpy()[shares.index.get_indexer(miss.event_id)]
                         * miss.loss.to_numpy()[:, None]).sum(0) / m
true_occ = oep_cotvar(account_plt)

def table(A_, O_, occ_):
    sa, so = np.sort(A_, 0), np.sort(O_, 0)
    return pd.DataFrame({"AEP AAL": A_.mean(0),
                         "AEP VaR": sa[-m], "AEP TVaR": sa[-m:].mean(0),
                         "AEP coTVaR": A_[aep_tail].mean(0),
                         "OEP VaR": so[-m], "OEP TVaR": so[-m:].mean(0),
                         "OEP coTVaR": occ_}, index=accounts)

true, recon = table(AEP_T, OEP_T, true_occ), table(AEP, OEP, rec_occ)
err = (recon - true).abs() / true.replace(0, np.nan) * 100
display(pd.concat({"true": true, "reconstructed": recon}, axis=1)
        .swaplevel(axis=1).sort_index(axis=1).round(2))
display(err.round(4).rename(columns=lambda c: c + " err %"))

AEP AAL             AEP TVaR               AEP VaR          \
       reconstructed   true reconstructed    true reconstructed    true   
ACC-01        31.320 31.320       670.670 697.840       398.000 447.540   
ACC-02        36.090 36.090       578.850 599.790       367.000 406.790   
ACC-03        17.440 17.440       262.990 280.840       177.550 187.060   
ACC-04        12.680 12.680       214.870 233.950       148.410 150.390   
ACC-05        58.010 58.010       937.200 953.760       608.660 667.070   
ACC-06         5.840  5.840        62.640  69.590        45.380  50.190   
ACC-07        32.790 32.790       536.110 560.690       356.570 374.130   
ACC-08        18.730 18.730       548.320 577.940       323.390 334.340   

          AEP coTVaR              OEP TVaR               OEP VaR          \
       reconstructed    true reconstructed    true reconstructed    true   
ACC-01       419.720 419.720       641.280 670.000       367.080 423.270   
ACC-02       481.740 481.740       542.210 564.020       332.040 372.240   
ACC-03        78.750  78.750       243.490 261.150       162.000 170.760   
ACC-04        49.000  49.000       200.520 218.330       137.500 138.200   
ACC-05       668.140 668.140       887.590 909.400       552.040 602.100   
ACC-06        23.730  23.730        54.890  62.130        39.430  44.070   
ACC-07       361.200 361.200       498.860 523.130       325.640 347.890   
ACC-08       203.790 203.790       529.190 559.730       305.210 318.350   

          OEP coTVaR          
       reconstructed    true  
ACC-01       387.160 387.160  
ACC-02       446.780 446.780  
ACC-03        62.190  62.190  
ACC-04        28.290  28.290  
ACC-05       630.560 630.560  
ACC-06        15.080  15.080  
ACC-07       321.580 321.580  
ACC-08       161.990 161.990

,AEP AAL err %,AEP VaR err %,AEP TVaR err %,AEP coTVaR err %,OEP VaR err %,OEP TVaR err %,OEP coTVaR err %
ACC-01,0.000,11.071,3.894,0.000,13.275,4.288,0.000
ACC-02,0.000,9.782,3.492,0.000,10.800,3.866,0.000
ACC-03,0.000,5.083,6.355,0.000,5.133,6.759,0.000
ACC-04,0.000,1.319,8.152,0.000,0.506,8.154,0.000
ACC-05,0.000,8.756,1.736,0.000,8.313,2.398,0.000
ACC-06,0.000,9.599,9.984,0.000,10.534,11.653,0.000
ACC-07,0.000,4.694,4.383,0.000,6.394,4.638,0.000
ACC-08,0.000,3.276,5.125,0.000,4.128,5.456,0.000


In [14]:
print("Portfolio level")
print(f"  AEP TVaR   true {np.sort(Y)[-m:].mean():>12,.2f}   recon {np.sort(AEP.sum(1))[-m:].mean():>12,.2f}")
print(f"  OEP TVaR   true {np.sort(X)[-m:].mean():>12,.2f}   recon {biggest.loss.mean():>12,.2f}")
print(f"  sum AEP coTVaR  true {true['AEP coTVaR'].sum():>10,.2f}   "
      f"recon {recon['AEP coTVaR'].sum():>10,.2f}   (= portfolio AEP TVaR)")
print(f"  sum OEP coTVaR  true {true['OEP coTVaR'].sum():>10,.2f}   "
      f"recon {recon['OEP coTVaR'].sum():>10,.2f}   (= portfolio OEP TVaR)")

print("\nWorst per-account error")
for c in err.columns:
    print(f"  {c:12s} {err[c].max():8.4f}%")

Portfolio level
  AEP TVaR   true     2,286.07   recon     2,286.07
  OEP TVaR   true     2,053.64   recon     2,053.64
  sum AEP coTVaR  true   2,286.07   recon   2,286.07   (= portfolio AEP TVaR)
  sum OEP coTVaR  true   2,053.64   recon   2,053.64   (= portfolio OEP TVaR)

Worst per-account error
  AEP AAL        0.0000%
  AEP VaR       11.0706%
  AEP TVaR       9.9838%
  AEP coTVaR     0.0000%
  OEP VaR       13.2752%
  OEP TVaR      11.6529%
  OEP coTVaR     0.0000%


## What survives

| metric | status | why |
|---|---|---|
| **AAL** | exact | shares fitted by summing then dividing, so each account's body total is reproduced identically |
| **AEP co-TVaR** | exact | reads only the portfolio's worst `Y` years, which are stored verbatim |
| **OEP co-TVaR** | exact | reads only the largest occurrence of each worst `X` year, also stored |
| **portfolio AEP / OEP** | exact | the portfolio table is kept whole |
| **per-account VaR / TVaR** | approximate | an account's own worst year is often a quiet year for the portfolio — a body year, where its loss is a share of the total rather than its real loss |

That last row is the trade. The compression is built for capital allocation, which reads the
portfolio's tail — not for per-account return periods, which read each account's own tail.

**OEP degrades more than AEP.** The share vector throws away how an event's split varies from one
occurrence to the next. A sum averages that noise out; a max is drawn to the occurrence where the
real split happened to favour an account, and that is exactly the variation the reconstruction has
flattened.

Three rules that keep it honest:

- `ALPHA >= Q_RETAIN`. A metric level outside the retained region is not covered.
- Rank on the **year**, not on individual occurrences.
- If OEP is reported, retain the **union** of the AEP and OEP tails.

## 6 · Marginal impact at a return period

Capital is usually set at a **return period** — 1-in-200 means `alpha = 1 - 1/200 = 0.995`. The
marginal impact of an account is its share of that capital.

Two ways to measure it, and they behave very differently:

**co-TVaR** — the account's average loss over the worst `m` years. Robust, because it averages many
years, and it sums exactly to portfolio TVaR.

**co-VaR** — the account's loss in the *single* year sitting at the VaR rank. On a finite table this
is one row. An account that happened not to lose in that one year gets allocated **zero capital**,
and the whole split reshuffles at the next model refresh. Standard errors below run to 90%+.

**So: use TVaR to decide the split, use VaR for the total.**

    marginal = co-TVaR proportions x portfolio VaR

It still sums exactly to VaR, but inherits co-TVaR's stability. This is standard practice for a
Solvency II style 99.5% VaR capital measure, not a workaround — VaR's own gradient conditions on a
probability-zero event, so a TVaR surrogate rescaled to the VaR total is the structurally correct
response.

The VaR year itself is inside the retained set (`ALPHA_RP >= Q_RETAIN`), so everything here is exact
under the compression — the noise is sampling noise in the underlying table, not reconstruction
error.

In [15]:
RP        = 200                      # return period
ALPHA_RP  = 1 - 1 / RP
m_rp      = int(round((1 - ALPHA_RP) * T_YEARS))
assert ALPHA_RP >= Q_RETAIN, f"RP {RP} needs alpha {ALPHA_RP}; retention only covers {Q_RETAIN}"

ord_Y   = np.argsort(-Y, kind="stable")
var_yr  = ord_Y[m_rp - 1]            # the SINGLE year sitting at the VaR rank
VaR_p   = Y[var_yr]
TVaR_p  = Y[ord_Y[:m_rp]].mean()

co_tvar  = AEP[ord_Y[:m_rp]].mean(0)          # average over the worst m years  -> robust
co_var   = AEP[var_yr]                        # that one year only              -> noisy
marginal = co_tvar / co_tvar.sum() * VaR_p    # TVaR split, VaR total           -> USE THIS

print(f"RP {RP} -> alpha {ALPHA_RP} | tail = {m_rp} of {T_YEARS:,} years")
print(f"portfolio VaR  {VaR_p:,.2f}   (year at rank {m_rp}, retained: {bool(is_kept[var_yr])})")
print(f"portfolio TVaR {TVaR_p:,.2f}")

RP 200 -> alpha 0.995 | tail = 100 of 20,000 years
portfolio VaR  2,174.31   (year at rank 100, retained: True)
portfolio TVaR 2,648.27


In [16]:
# --- bootstrap: resample years, recompute, report the spread ---------------
rb, B = np.random.default_rng(5), 400
bs_t, bs_v = np.empty((B, N_ACC)), np.empty((B, N_ACC))
for b in range(B):
    idx = rb.integers(0, T_YEARS, T_YEARS)
    Yb, Ab = Y[idx], AEP[idx]
    ob = np.argsort(-Yb, kind="stable")
    bs_t[b] = Ab[ob[:m_rp]].mean(0)
    bs_v[b] = Ab[ob[m_rp - 1]]
se_marg = bs_t.std(0) / co_tvar.sum() * VaR_p
se_cov  = bs_v.std(0)

res = pd.DataFrame({
    "MARGINAL (use this)": marginal,
    "SE":                  se_marg,
    "SE %":                100 * se_marg / marginal,
    "capital share %":     100 * marginal / marginal.sum(),
    "co-VaR (1 year)":     co_var,
    "co-VaR SE %":         100 * se_cov / np.where(co_var > 0, co_var, np.nan),
}, index=accounts)
display(res.round(2))

print(f"sum of MARGINAL {marginal.sum():,.2f} = portfolio VaR {VaR_p:,.2f}  "
      f"(diff {abs(marginal.sum()-VaR_p):.2e})")
print(f"median SE:  MARGINAL {np.nanmedian(res['SE %']):.0f}%   "
      f"raw co-VaR {np.nanmedian(res['co-VaR SE %']):.0f}%")
n0 = int((co_var == 0).sum())
if n0:
    print(f"\n{n0} of {N_ACC} accounts get ZERO capital under raw co-VaR - they simply did not lose")
    print("in that one year. That is the reason for rescaling rather than reading VaR directly.")

,MARGINAL (use this),SE,SE %,capital share %,co-VaR (1 year),co-VaR SE %
ACC-01,355.440,34.980,9.840,16.350,783.470,50.730
ACC-02,477.050,30.730,6.440,21.940,373.450,51.790
ACC-03,79.640,9.360,11.750,3.660,0.000,NaN
ACC-04,44.240,6.040,13.650,2.030,0.000,NaN
ACC-05,670.490,54.120,8.070,30.840,786.500,55.010
ACC-06,20.880,3.080,14.740,0.960,0.000,NaN
ACC-07,354.340,24.870,7.020,16.300,97.840,275.740
ACC-08,172.230,24.080,13.980,7.920,133.040,212.800


sum of MARGINAL 2,174.31 = portfolio VaR 2,174.31  (diff 4.55e-13)
median SE:  MARGINAL 11%   raw co-VaR 55%

3 of 8 accounts get ZERO capital under raw co-VaR - they simply did not lose
in that one year. That is the reason for rescaling rather than reading VaR directly.


### Reading this

`MARGINAL` is the number to price and allocate off. It sums exactly to portfolio VaR and its
bootstrap SE is a fraction of raw co-VaR's.

**Watch the SEs even so.** At RP 200 the tail is `T/200` years — 100 here, but only 50 if you run
10,000 trials. A small or diversifying account's marginal is the jumpiest figure on the page, and a
move between model runs that sits inside its own SE is not a signal. Circulate the SE beside every
allocated number.

**This is a marginal price at today's mix**, correct for "what should I charge for this account" or
"what does this renewal cost me". It is not a budget for a large move: write three times as much of a
diversifying account and it consumes its own diversification credit, so the number shifts. For
anything that changes the book materially, recompute at the target mix rather than scaling this one.

## Switching input mode

`INPUT_MODE = "PLT"` skips §0 entirely — point cell §1 at your own `read_parquet` calls and every
other cell runs unchanged. `INPUT_MODE = "ELT"` runs the engine and derives the PLTs from it.

**If you scale up the ELT path**, check `elt.groupby("event_id").size().describe()` first. The engine
loops events in Python and vectorises across occurrences within each, so a catalogue of tens of
thousands of events with narrow footprints runs slower per row than a smaller catalogue with wide
ones. If that bites, parallelise the **event** loop — never the account loop inside it, whose members
must consume the same `z_shared`.